# Codebook-Free Beamforming - Weeks 1 to 4 (run everything)

This notebook runs all three members' code top to bottom on Colab's free GPU.

**Before you start:** menu -> Runtime -> Change runtime type -> **T4 GPU** -> Save.

Then run every cell in order (Runtime -> Run all). Read the short note above each
cell so you know what it is proving.

## Step 0 - get the code and install the libraries

In [ ]:
# This repo is PRIVATE, so Colab needs a GitHub token to download it.
# ONE-TIME setup:
#   1. Make a token: GitHub -> Settings -> Developer settings ->
#      Personal access tokens -> Fine-grained tokens -> Generate new token
#      -> give it 'Contents: Read-only' access to this repo.
#   2. In Colab: click the key icon (Secrets) on the left bar -> Add new secret
#      -> name it  GH_TOKEN , paste the token, turn ON 'Notebook access'.
# (Or just make the repo public on GitHub and skip all this.)
import os
REPO = 'sneha822/codebook-free-beamforming'
FOLDER = REPO.split('/')[1]
if not os.path.exists(FOLDER):
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    if tok:
        !git clone https://{tok}@github.com/{REPO}.git
    else:
        print('No GH_TOKEN secret found. Add it (see the steps above),')
        print('or make the repo public, or upload the folder manually.')
if os.path.exists(FOLDER):
    %cd {FOLDER}
    !pip -q install torch numpy matplotlib
    print('\nready - now run the next cells in order')

## Member 1 - Wireless (the data and the reference scores)

### Week 2 result: make the dataset
Creates `positions.npy` (where each user is) and `channels.npy` (how the signal
reaches them). This is the raw material everything else needs.

In [ ]:
!cd member1 && python generate_channels.py

### Week 1 + 3 result: see the street and the dead zones
Left = where users are. Right = signal strength (dark areas are weak "dead zones",
which later explain where the AI struggles).

In [ ]:
!cd member1 && python plot_scenario.py
from IPython.display import Image
Image('member1/scenario.png')

### Week 4 result: the baselines (the lines our AI must sit between)
The phase-only upper bound is the best score possible (our ceiling). The DFT
codebook is the old method (the floor we must beat).

In [ ]:
!cd member1 && python baselines.py

## Member 2 - DL Engineer (the AI and the training machine)

### Week 1 result: the tools work
Trains a throwaway network on random data. We only care that the loss drops -
that proves PyTorch and the GPU are working.

In [ ]:
!cd member2 && python w1_sandbox.py

### Week 2 result: data flows in correct batches
Loads Member 1's data, splits complex channels into real/imag, normalises, and
prints the batch shapes. 'mean power ~1' means the normalisation worked.

In [ ]:
!cd member2 && python dataset.py

### Week 3 result: the model does a forward pass
BeamNet takes a position and outputs M phase values. Output shape (512, 64) =
512 users, 64 antenna settings each.

In [ ]:
!cd member2 && python model.py

## Member 3 - Maths (the scorekeeper and the proofs)

### Week 2 result: real outputs become valid phases
Proves that for ANY network output, every antenna ends up at equal power
(constant modulus) and total power is exactly 1 - so no clipping is ever needed.

In [ ]:
!cd member3 && python phase_mapping_demo.py

### Week 3 result: the loss runs and is differentiable
Computes the spectral-efficiency loss on dummy data. A negative number is
correct, and the gradient flows (no autograd error).

In [ ]:
!cd member3 && python loss.py

### Week 4 result: the maths is formally correct
`gradcheck passed: True` means the loss's gradients are provably correct, so
Member 2 can trust training. It also confirms the known best-case phases reach
the theoretical optimum.

In [ ]:
!cd member3 && python gradient_check.py

## Member 2 - Week 4 THE MILESTONE: everything together

This uses Member 1's data + Member 3's loss + Member 2's model at once. It proves
gradients flow from the physics loss back to the weights, the loss drops after one
step, and the **gap to the best-possible rate shrinks each epoch** - that shrinking
gap is the real proof that the AI is learning.

In [ ]:
!cd member2 && python train.py

## Done
If every cell ran and the gap above got smaller each epoch, Weeks 1-4 are
complete and the full pipeline works. Next: Week 5 (overfitting 10 samples to
prove learning), then real training in Week 6.